# Multi-Source Data Merge

=============================================================
 Fashion Sustainability Dataset — Multi-Source Merge Script
=============================================================
 Purpose: Merge WikiRate FTI 2024 + CDP Scores + SBTi Registry
          + Remake 2021 + Manual region/subsegment tags into
          one master dataset (Fashion_Sustainability_Final_Dataset.csv)

 Inputs:
   - Fashion_Dataset.csv           (WikiRate + CDP scores, pre-collected)
   - SBTI_Data.xlsx                (SBTi registry, 14,531 companies)
   - Remake scores                 (manually extracted from PDF)
   - Region + Sub-segment          (manual classification)

 Output:
   - Fashion_Sustainability_Final_Dataset.csv (98 columns, 64 companies)
=============================================================


In [ ]:
import pandas as pd
import numpy as np

## FILE PATHS


In [ ]:
FASHION_CSV  = 'Fashion_Dataset.csv'
SBTI_XLSX    = 'SBTI_Data.xlsx'
OUTPUT_CSV   = 'Fashion_Sustainability_Final_Dataset.csv'

## LOAD MAIN DATASET (WikiRate + CDP already merged)


In [ ]:
print("Loading main dataset...")
main_df = pd.read_csv(FASHION_CSV)
print(f"  Shape: {main_df.shape}")

## LOAD SBTi DATA


In [ ]:
print("Loading SBTi registry...")
sbti = pd.read_excel(SBTI_XLSX)
print(f"  SBTi shape: {sbti.shape}")

## MANUAL SBTi COMPANY NAME MAPPINGS


In [ ]:
# Maps exact company names in our dataset to search terms in SBTi
company_mappings = {
    'Puma': 'Puma', 'H&M': 'H & M', 'Hanesbrands': 'Hanesbrands',
    'lululemon athletica': 'lululemon', 'Adidas AG': 'adidas',
    'Asics Corporation': 'ASICS', 'Ralph Lauren Corporation': 'Ralph Lauren',
    'Gildan Activewear Inc.': 'Gildan', 'Superdry plc': 'Superdry',
    'Levi Strauss & Co.': 'Levi Strauss', 'Nike Inc.': 'Nike',
    'JD Sports Fashion plc': 'JD Sports', 'Gap inc.': 'Gap Inc',
    'Hermes International': 'Hermes', 'Hugo Boss AG': 'Hugo Boss',
    'Esprit Holdings Limited': 'Esprit', 'Prada': 'Prada',
    'Guess? Inc': 'Guess', 'Next PLC': 'Next', 'Zalando SE': 'Zalando',
    'Asos': 'ASOS', 'American Eagle Outfitters': 'American Eagle',
    'Moncler': 'Moncler', 'Target': 'Target Corporation',
    'Marks and Spencer Group plc': 'Marks and Spencer',
    'Abercrombie & Fitch': 'Abercrombie',
    'Capri Holdings Ltd (formerly Michael Kors)': 'Capri Holdings',
    'Amazon.com, Inc.': 'Amazon', 'Salvatore Ferragamo SpA': 'Salvatore Ferragamo',
    'Nordstrom': 'Nordstrom', 'Columbia Sportswear': 'Columbia Sportswear',
    'Burberry Group plc': 'Burberry', "Macy's": 'Macy',
    'Ted Baker': 'Ted Baker', "Dick's Sporting Goods": 'Dicks Sporting',
    'Boohoo.com': 'boohoo', 'Otto Group': 'Otto Group',
    'Louis Vuitton Malletier SA (LVMH)': 'LVMH', 'Aldi Nord': 'Aldi',
    'Walmart': 'Walmart', 'Fossil Group, Inc.': 'Fossil Group',
    'Under Armour': 'Under Armour', "Carter's Inc": 'Carter',
    "Children's Place Inc": 'Children', 'Carrefour S.A.': 'Carrefour',
    'Canada Goose': 'Canada Goose', 'Anta Sports Products': 'ANTA',
    'Costco Wholesale': 'Costco', 'Foot Locker Inc.': 'Foot Locker',
    'Burlington Stores Inc': 'Burlington', 'Gerry Weber': 'Gerry Weber',
    'Sports Direct': 'Sports Direct', 'Urban Outfitters': 'Urban Outfitters',
    'Ross Stores': 'Ross Stores', 'El Corte Ingles S.A.': 'El Corte',
    "Chico's FAS Inc": 'Chico', "TOD'S": 'Tod',
    'Brunello Cucinelli': 'Brunello', 'Express Inc': 'Express',
    'Buckle Inc': 'Buckle', 'DSW Inc.': 'DSW',
    "Dillard's, Inc.": 'Dillard', 'Skechers USA Inc': 'Skechers', 'Semir': 'Semir',
}

## MATCH SBTi DATA


In [ ]:
print("Matching SBTi data...")
sbti_rows = []
for company, search_term in company_mappings.items():
    mask = sbti['company_name'].str.lower().str.contains(search_term.lower(), na=False)
    matches = sbti[mask]
    if len(matches) > 0:
        row = matches.iloc[0]
        near = str(row['near_term_status']).strip() if pd.notna(row['near_term_status']) else 'Not Found'
        net_zero = str(row['net_zero_status']).strip() if pd.notna(row['net_zero_status']) else 'None'
        nz_year = row['net_zero_year'] if pd.notna(row['net_zero_year']) else None
        nt_year = row['near_term_target_year'] if pd.notna(row['near_term_target_year']) else None
    else:
        near, net_zero, nz_year, nt_year = 'Not Found', 'None', None, None
    sbti_rows.append({
        'Company': company,
        'SBTi_Near_Term_Status': near,
        'SBTi_Net_Zero_Status': net_zero,
        'SBTi_Net_Zero_Year': nz_year,
        'SBTi_Near_Term_Target_Year': nt_year,
    })

sbti_df = pd.DataFrame(sbti_rows)

# Numeric encoding
sbti_numeric = {'Targets set': 3, 'Committed': 2, 'Commitment removed': 1, 'Not Found': 0}
sbti_df['SBTi_Near_Term_Numeric'] = sbti_df['SBTi_Near_Term_Status'].map(
    lambda x: sbti_numeric.get(str(x).strip(), 0))
sbti_df['Has_Net_Zero_Target'] = sbti_df['SBTi_Net_Zero_Status'].apply(
    lambda x: 1 if str(x).strip() == 'Targets set' else 0)

matched = (sbti_df['SBTi_Near_Term_Status'] != 'Not Found').sum()
print(f"  Matched: {matched}/{len(sbti_df)} companies")

## REMAKE 2021 SCORES (manually extracted from PDF)


In [ ]:
remake_scores = {
    'Abercrombie & Fitch': -2, 'Adidas AG': 25, 'Amazon.com, Inc.': 2,
    'Asos': 16, 'Boohoo.com': 13, 'Burberry Group plc': 38,
    'Gap inc.': 6, 'H&M': 39, 'Levi Strauss & Co.': 20,
    'Marks and Spencer Group plc': 31, 'Nike Inc.': 25, 'Next PLC': 13,
    'Ross Stores': -13, 'Target': 1, 'Under Armour': 0,
    'Urban Outfitters': 3, 'Walmart': -1, 'Zalando SE': 18,
    'lululemon athletica': -3, 'American Eagle Outfitters': -4,
    "Children's Place Inc": -10,
}
remake_df = pd.DataFrame([
    {'Company': k, 'Remake_2021_Score_out_of_150': v}
    for k, v in remake_scores.items()
])

## REGION MAPPING (manual, based on HQ location)


In [ ]:
region_map = {
    'Puma': 'Europe', 'H&M': 'Europe', 'Hanesbrands': 'North America',
    'lululemon athletica': 'North America', 'Adidas AG': 'Europe',
    'Asics Corporation': 'Asia', 'Ralph Lauren Corporation': 'North America',
    'Gildan Activewear Inc.': 'North America', 'Superdry plc': 'Europe',
    'Levi Strauss & Co.': 'North America', 'Nike Inc.': 'North America',
    'JD Sports Fashion plc': 'Europe', 'Gap inc.': 'North America',
    'Hermes International': 'Europe', 'Hugo Boss AG': 'Europe',
    'Esprit Holdings Limited': 'Asia', 'Prada': 'Europe',
    'Guess? Inc': 'North America', 'Next PLC': 'Europe',
    'Zalando SE': 'Europe', 'Asos': 'Europe',
    'American Eagle Outfitters': 'North America', 'Moncler': 'Europe',
    'Target': 'North America', 'Marks and Spencer Group plc': 'Europe',
    'Abercrombie & Fitch': 'North America',
    'Capri Holdings Ltd (formerly Michael Kors)': 'North America',
    'Amazon.com, Inc.': 'North America', 'Salvatore Ferragamo SpA': 'Europe',
    'Nordstrom': 'North America', 'Columbia Sportswear': 'North America',
    'Burberry Group plc': 'Europe', "Macy's": 'North America',
    'Ted Baker': 'Europe', "Dick's Sporting Goods": 'North America',
    'Boohoo.com': 'Europe', 'Otto Group': 'Europe',
    'Louis Vuitton Malletier SA (LVMH)': 'Europe', 'Aldi Nord': 'Europe',
    'Walmart': 'North America', 'Fossil Group, Inc.': 'North America',
    'Under Armour': 'North America', "Carter's Inc": 'North America',
    "Children's Place Inc": 'North America', 'Carrefour S.A.': 'Europe',
    'Canada Goose': 'North America', 'Anta Sports Products': 'Asia',
    'Costco Wholesale': 'North America', 'Foot Locker Inc.': 'North America',
    'Burlington Stores Inc': 'North America', 'Gerry Weber': 'Europe',
    'Sports Direct': 'Europe', 'Urban Outfitters': 'North America',
    'Ross Stores': 'North America', 'El Corte Ingles S.A.': 'Europe',
    "Chico's FAS Inc": 'North America', "TOD'S": 'Europe',
    'Brunello Cucinelli': 'Europe', 'Express Inc': 'North America',
    'Buckle Inc': 'North America', 'DSW Inc.': 'North America',
    "Dillard's, Inc.": 'North America', 'Skechers USA Inc': 'North America',
    'Semir': 'Asia',
}
region_df = pd.DataFrame([{'Company': k, 'HQ_Region': v} for k, v in region_map.items()])

## CDP NUMERIC ENCODING


In [ ]:
cdp_numeric = {
    'A': 7, 'A-': 6, 'B': 5, 'B-': 4,
    'C': 3, 'C-': 2, 'D': 1, 'D-': 0,
    'F': 0, 'Not Disclosed': None
}
main_df['CDP Score'] = main_df['CDP Score'].replace({'Nil': 'Not Disclosed', 'nil': 'Not Disclosed'})
main_df['CDP_Score_Numeric'] = main_df['CDP Score'].map(cdp_numeric)

## MERGE ALL SOURCES


In [ ]:
print("Merging all sources...")
merged = main_df.copy()
merged = merged.merge(sbti_df,   on='Company', how='left')
merged = merged.merge(remake_df, on='Company', how='left')
merged = merged.merge(region_df, on='Company', how='left')

## SAVE


In [ ]:
merged.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ Saved: {OUTPUT_CSV}")
print(f"   Shape: {merged.shape}")
print(f"   Companies: {merged['Company'].nunique()}")